# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library. We'll walk through loading the metadata and records, inspecting record sets and fields (using their `@id`s), and performing basic exploratory data analysis.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata  # Not subscriptable; treat as an object

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, columns, and their `@id` identifiers.

We'll list all record sets (`cr:RecordSet`) and show their field and column `@id`s.

In [ ]:
# List all record sets and their identifiers.
print('Available RecordSets in the dataset:')
record_set_objs = dataset.record_sets  # This is a dict: {record_set_id: RecordSet}
for rs_id, rs in record_set_objs.items():
    print(f"- RecordSet @id: {rs_id}")
    print(f"  Name: {getattr(rs, 'name', 'N/A')}")
    if hasattr(rs, 'fields'):
        print("  Fields:")
        for field_id, field_obj in rs.fields.items():
            print(f"      @id: {field_id}, name: {getattr(field_obj, 'name', 'N/A')}, type: {getattr(field_obj, 'dataType', 'N/A')}")
    if hasattr(rs, 'columns'):
        print("  Columns:")
        for col_id, col_obj in rs.columns.items():
            print(f"      @id: {col_id}, name: {getattr(col_obj, 'name', 'N/A')}, type: {getattr(col_obj, 'dataType', 'N/A')}")
    print()

## 3. Data Extraction

Now we'll extract data from record sets into Pandas DataFrames. All entities are referenced by their `@id` fields for consistency.

We first choose the main record set (by `@id`) containing the tabular clinical and pathological data.

In [ ]:
# List all record set IDs for extraction
record_sets_ids = list(dataset.record_sets.keys())
# Print them for reference
print('RecordSet @ids:', record_sets_ids)

# For this dataset, assuming only one major record set exists with clinical data; pick the first
main_record_set_id = record_sets_ids[0]  # If multiple, adapt as needed
print('Selected RecordSet for extraction:', main_record_set_id)

# Load all record sets into DataFrames (by @id)
dataframes = {}
for rs_id in record_sets_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:  # Only create a DataFrame if there is data
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for RecordSet {rs_id}: shape {dataframes[rs_id].shape}")
    else:
        print(f"No records found for RecordSet {rs_id}")

# Show columns (fields) and head of the main dataframe
if main_record_set_id in dataframes:
    print("\nField @ids in main DataFrame:")
    print(list(dataframes[main_record_set_id].columns))
    display(dataframes[main_record_set_id].head())
else:
    print("No data loaded for main record set.")

## 4. Exploratory Data Analysis (EDA)

We'll analyze a numeric field, filter the table, perform normalization, and group by a categorical field. All fields are referenced by their full `@id`.

**Note:** Adjust the `numeric_field_id` and `group_field_id` if columns differ. To discover field IDs, check the record set details above.

For this dataset, let's pick patient Age for analysis if available (assume `cr:PatientAge` for example), and group by Sex (e.g., `cr:Sex`).

In [ ]:
# If you do not know the field @ids, inspect them below:
main_df = dataframes[main_record_set_id]
print('Available fields (@ids):', list(main_df.columns))

# These are EXAMPLE field @ids. Replace by those present in the dataset!
# E.g., 'cr:PatientAge' and 'cr:Sex' as plausible identifiers.
numeric_field_id = None
group_field_id = None
for col in main_df.columns:
    # Try to pick a likely numeric field (anything with 'Age', 'Interval', 'Metastasis', or 'Count')
    if (not numeric_field_id) and (any(x in col.lower() for x in ['age', 'interval', 'count'])):
        numeric_field_id = col
    # Pick a plausible group field ('Sex', 'Location', 'Subtype', etc.)
    if (not group_field_id) and (any(x in col.lower() for x in ['sex', 'location', 'type', 'msi'])):
        group_field_id = col

print(f"Numeric field for EDA: {numeric_field_id}")
print(f"Group field: {group_field_id}")

# Check the type and content of the numeric field
if numeric_field_id is not None and np.issubdtype(main_df[numeric_field_id].dropna().dtype, np.number):
    threshold = main_df[numeric_field_id].mean()  # For demo: filter above mean
else:
    # Try to convert to numeric (may be object due to load)
    main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')
    threshold = main_df[numeric_field_id].mean()

# Filter records above threshold
filtered_df = main_df[main_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize numeric field (z-score)
norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"First normalized values for {numeric_field_id}:")
display(filtered_df[[numeric_field_id, norm_col]].head())

# Group by group_field_id and calculate mean of numeric field
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
    display(grouped_df.head())
else:
    print(f"Group field {group_field_id} not found in columns.")

## 5. Visualization

Visualize the distribution of the selected numeric field and its relationship with the group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(main_df[numeric_field_id].dropna(), bins=15, kde=True)
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.title(f"Distribution of {numeric_field_id}")
plt.show()

# Boxplot of numeric field grouped by group_field_id (if available)
if group_field_id in main_df.columns:
    plt.figure(figsize=(7,4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=main_df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion

In this notebook, we loaded and explored the FAIR² colorectal cancer survivor dataset using `mlcroissant`. By referencing all entities via their `@id` fields, we extracted metadata, examined record sets and fields, and performed exploratory analysis of a numeric variable, grouped by a clinical attribute. This workflow enables robust and reproducible FAIR-aware data science for clinical datasets using Croissant schemas.